# Level 3: HIGH  -  Tuning, Kelas Tidak Seimbang, dan Interpretasi Model

**Estimasi waktu:** 3-4 jam

**Prasyarat:** Selesaikan `02_level_medium.ipynb` terlebih dahulu.

## Tujuan Pembelajaran

Setelah menyelesaikan level ini, Anda akan mampu:

1. Melakukan **hyperparameter tuning** dengan GridSearchCV
2. Menangani **kelas tidak seimbang** (imbalanced classes)
3. Membaca dan menginterpretasi **kurva ROC** dan **Precision-Recall**
4. Menggunakan **Random Forest** sebagai ensemble model
5. Memahami **feature importance** dengan permutation importance
6. Membandingkan beberapa model secara **sistematis**

---

## Teori: Dari Satu Model ke Sistem Evaluasi yang Andal

Di Level Medium, kita belajar membuat dan mengevaluasi model dasar. Di level ini, kita mendalami:

### 1. Hyperparameter Tuning

Model machine learning memiliki **hyperparameter**  -  pengaturan yang TIDAK dipelajari dari data, tapi ditentukan oleh kita.

```
Contoh hyperparameter:
  KNN       -> n_neighbors (K), metric (euclidean/manhattan)
  Tree      -> max_depth, min_samples_split, min_samples_leaf
  LogReg    -> C (regularization strength), penalty (l1/l2)
  RandomForest -> n_estimators, max_depth, max_features
```

**GridSearchCV** mencoba SEMUA kombinasi dan memilih yang terbaik via cross-validation:

```
Hyperparameter Grid:
  C = [0.01, 0.1, 1, 10]
  penalty = [l1, l2]
  -> 4 × 2 = 8 kombinasi × 5 fold = 40 kali fitting!
```

### 2. Kelas Tidak Seimbang

Banyak masalah dunia nyata memiliki kelas yang tidak seimbang:

```
Deteksi fraud:    99% normal, 1% fraud
Deteksi kanker:   95% sehat, 5% sakit
Churn pelanggan:  85% stay, 15% churn

 Jika model selalu prediksi "normal", accuracy = 99%!
   Tapi model TIDAK BERGUNA karena tidak bisa mendeteksi fraud.
```

Solusi:
- **`class_weight='balanced'`**: memberikan bobot lebih besar ke kelas minoritas
- **Metrik alternatif**: balanced accuracy, F1, precision-recall AUC
- **Resampling** (SMOTE, undersampling)  -  tidak dibahas di sini, tapi worth exploring

### 3. Kurva ROC dan Precision-Recall

Model klasifikasi menghasilkan **probabilitas**. Kita bisa mengatur **threshold** (ambang batas) untuk mengubah probabilitas menjadi label:

```
Threshold tinggi (0.9): Precision tinggi, tapi recall rendah
  -> "Saya hanya bilang positif kalau sangat yakin"
  
Threshold rendah (0.1): Recall tinggi, tapi precision rendah  
  -> "Saya deteksi sebanyak mungkin, walau banyak alarm palsu"

Kurva ROC:  Trade-off TPR vs FPR di berbagai threshold
Kurva PR:   Trade-off Precision vs Recall (lebih informatif saat kelas tidak seimbang)
```

## 0. Persiapan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    average_precision_score, f1_score, make_scorer
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

print(' Semua library berhasil dimuat!')

---

## 1. Dataset: Breast Cancer Wisconsin

Dataset ini berisi 569 sampel dengan 30 fitur pengukuran sel tumor.
- **Target**: 0 = malignant (ganas), 1 = benign (jinak)
- **Masalah**: klasifikasi biner
- **Distribusi**: ~63% benign, ~37% malignant (sedikit tidak seimbang)

In [ ]:
# Muat dataset
cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

print(f'Ukuran data: {X.shape}')
print(f'Jumlah fitur: {X.shape[1]}')
print(f'Target: 0={cancer.target_names[0]}, 1={cancer.target_names[1]}')
print()

# Distribusi target
print(' Distribusi target:')
dist = y.value_counts()
dist_pct = y.value_counts(normalize=True).mul(100).round(1)
target_df = pd.DataFrame({
    'kelas': [cancer.target_names[i] for i in dist.index],
    'jumlah': dist.values,
    'persentase (%)': dist_pct.values
})
display(target_df)

ratio = dist.min() / dist.max()
print(f'\nRasio minoritas/mayoritas: {ratio:.2f}')
if ratio < 0.5:
    print(' Kelas cukup tidak seimbang  -  perlu perhatian khusus!')
else:
    print('Catatan: Ketidakseimbangan moderat  -  gunakan balanced accuracy dan class_weight.')

In [ ]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print(f'Training set: {X_train.shape[0]} sampel')
print(f'Test set    : {X_test.shape[0]} sampel')
print(f'Proporsi target train: {y_train.value_counts(normalize=True).sort_index().round(3).to_dict()}')
print(f'Proporsi target test : {y_test.value_counts(normalize=True).sort_index().round(3).to_dict()}')

---

## 2. Baseline dan Model Pertama

Kita mulai dengan baseline, lalu Logistic Regression TANPA tuning sebagai model awal.

In [ ]:
# Baseline
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, dummy.predict(X_test))
baseline_bal = balanced_accuracy_score(y_test, dummy.predict(X_test))
print(f'Baseline accuracy         : {baseline_acc:.3f}')
print(f'Baseline balanced accuracy : {baseline_bal:.3f}')
print()
print(' Balanced accuracy menghitung rata-rata recall per kelas.')
print('   Baseline balanced accuracy selalu 0.50 (50%) untuk 2 kelas.')
print('   Ini lebih adil daripada accuracy saat kelas tidak seimbang.')

In [ ]:
# Logistic Regression TANPA tuning (sebagai perbandingan)
lr_basic = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])
lr_basic.fit(X_train, y_train)
y_pred_basic = lr_basic.predict(X_test)

print('Logistic Regression (TANPA tuning, TANPA class_weight):')
print(f'  Accuracy         : {accuracy_score(y_test, y_pred_basic):.3f}')
print(f'  Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_basic):.3f}')
print()
print(classification_report(y_test, y_pred_basic, 
                            target_names=cancer.target_names, zero_division=0))

---

## 3. Hyperparameter Tuning dengan GridSearchCV

### Teori: Cara Kerja GridSearchCV

```
GridSearchCV Workflow:

1. Definisikan grid parameter:  C = [0.01, 0.1, 1, 10]

2. Untuk setiap kombinasi parameter:
   ┌──────────────────────────────────────────┐
   │ C = 0.01                                  │
   │   Fold 1: train(fold2-5) -> eval(fold1)   │
   │   Fold 2: train(fold1,3-5) -> eval(fold2) │
   │   ...                                     │
   │   Fold 5: train(fold1-4) -> eval(fold5)   │
   │   -> Rata-rata skor: 0.91                 │
   ├──────────────────────────────────────────┤
   │ C = 0.1  -> Rata-rata skor: 0.94          │
   ├──────────────────────────────────────────┤
   │ C = 1    -> Rata-rata skor: 0.96  <- BEST │
   ├──────────────────────────────────────────┤
   │ C = 10   -> Rata-rata skor: 0.95          │
   └──────────────────────────────────────────┘

3. Pilih parameter terbaik -> refit di seluruh training data
4. Evaluasi SATU KALI di test data
```

> ** Penting:** GridSearchCV harus menerima **Pipeline**, bukan data yang sudah di-scale. Jika tidak, scaler melihat data validasi -> data leakage!

In [ ]:
# GridSearchCV untuk Logistic Regression
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=3000, 
        class_weight='balanced',   # <- penting untuk kelas tidak seimbang!
        random_state=RANDOM_STATE
    ))
])

# Definisikan grid parameter
# Nama parameter: 'langkah__parameter' -> 'model__C'
param_grid_lr = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_lr = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid_lr,
    scoring='balanced_accuracy',  # <- metrik untuk optimasi
    cv=cv,
    n_jobs=-1,            # gunakan semua CPU
    return_train_score=True  # untuk mendeteksi overfit
)

grid_lr.fit(X_train, y_train)

print('='*60)
print('HASIL GRID SEARCH: Logistic Regression')
print('='*60)
print(f'Parameter terbaik: {grid_lr.best_params_}')
print(f'CV balanced accuracy terbaik: {grid_lr.best_score_:.3f}')

# Tampilkan semua hasil
results_lr = pd.DataFrame(grid_lr.cv_results_)
cols = ['param_model__C', 'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']
print()
display(results_lr[cols].sort_values('rank_test_score').round(4))

In [ ]:
# Visualisasi: pengaruh C terhadap skor (train vs validation)
fig, ax = plt.subplots(figsize=(10, 5))

C_values = results_lr['param_model__C'].astype(float)
train_scores = results_lr['mean_train_score']
val_scores = results_lr['mean_test_score']
val_stds = results_lr['std_test_score']

ax.semilogx(C_values, train_scores, 'o-', label='Training Score', color='steelblue')
ax.semilogx(C_values, val_scores, 'o-', label='Validation Score', color='darkorange')
ax.fill_between(C_values, val_scores - val_stds, val_scores + val_stds, 
                alpha=0.2, color='darkorange')

best_C = grid_lr.best_params_['model__C']
ax.axvline(x=best_C, linestyle='--', color='red', alpha=0.7, label=f'C terbaik = {best_C}')

ax.set_xlabel('C (regularization parameter)')
ax.set_ylabel('Balanced Accuracy')
ax.set_title('Validation Curve: Logistic Regression', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(' C kecil = regularisasi kuat (model lebih sederhana)')
print('   C besar = regularisasi lemah (model lebih kompleks)')
print('   Gap besar train-val = tanda overfit')

---

## 4. Random Forest: Ensemble Model

### Teori: Dari Satu Tree ke Forest

**Random Forest** = banyak Decision Tree yang di-vote.

```
Dataset Training
      │
      ├---> Bootstrap Sample 1 ---> Tree 1 ---> Prediksi: Benign
      ├---> Bootstrap Sample 2 ---> Tree 2 ---> Prediksi: Malignant
      ├---> Bootstrap Sample 3 ---> Tree 3 ---> Prediksi: Benign
      ├---> ...                                    ...
      └---> Bootstrap Sample N ---> Tree N ---> Prediksi: Benign
                                                  │
                                    Voting Mayoritas: BENIGN 
```

**Kelebihan dibanding satu tree:**
- Lebih stabil (variance lebih rendah)
- Lebih jarang overfit
- Bisa mengukur feature importance

**Key hyperparameters:**
- `n_estimators`: jumlah pohon (lebih banyak = lebih stabil, tapi lebih lambat)
- `max_depth`: kedalaman maksimum setiap pohon
- `max_features`: jumlah fitur yang dipertimbangkan di setiap split

In [ ]:
# GridSearchCV untuk Random Forest
rf_pipeline = Pipeline([
    # Random Forest TIDAK perlu scaling (tree-based)
    ('model', RandomForestClassifier(
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

param_grid_rf = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 5, 10, None],
    'model__max_features': ['sqrt', 'log2'],
}

# Total kombinasi: 3 × 4 × 2 = 24 × 5 fold = 120 kali fitting
print(f'Total kombinasi: {3*4*2} × 5 fold = {3*4*2*5} fits')
print('Mohon tunggu...')

grid_rf = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid_rf,
    scoring='balanced_accuracy',
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_rf.fit(X_train, y_train)

print(f'\nParameter terbaik: {grid_rf.best_params_}')
print(f'CV balanced accuracy terbaik: {grid_rf.best_score_:.3f}')

# Top 5 kombinasi
results_rf = pd.DataFrame(grid_rf.cv_results_)
top5 = results_rf.nsmallest(5, 'rank_test_score')[[
    'param_model__n_estimators', 'param_model__max_depth', 
    'param_model__max_features', 'mean_test_score', 'std_test_score', 'rank_test_score'
]]
print('\nTop 5 kombinasi parameter:')
display(top5.round(4))

---

## 5. Evaluasi Akhir pada Data Test

### Teori: Evaluasi yang Benar

```
Alur yang BENAR:

Training Data ---> GridSearchCV (pilih parameter) ---> Best Model
                                                         │
Test Data ────────────────────────────────────---> Evaluasi SATU KALI

 JANGAN: Tuning di test data -> evalusi di test data (double dipping!)
```

In [ ]:
# Evaluasi kedua model terbaik pada data test
best_lr = grid_lr.best_estimator_
best_rf = grid_rf.best_estimator_

models_final = {
    'Baseline (majority)': dummy,
    'Logistic Regression (tuned)': best_lr,
    'Random Forest (tuned)': best_rf,
}

print('='*65)
print('EVALUASI AKHIR PADA DATA TEST')
print('='*65)

final_results = []
for name, model in models_final.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    final_results.append({
        'Model': name,
        'Accuracy': acc,
        'Balanced Acc': bal_acc,
        'F1 (weighted)': f1
    })

final_df = pd.DataFrame(final_results).set_index('Model')
display(final_df.round(3))

In [ ]:
# Classification report detail untuk model terbaik
print('Classification Report  -  Logistic Regression (tuned):')
print(classification_report(y_test, best_lr.predict(X_test), 
                            target_names=cancer.target_names, zero_division=0))

print('Classification Report  -  Random Forest (tuned):')
print(classification_report(y_test, best_rf.predict(X_test), 
                            target_names=cancer.target_names, zero_division=0))

In [ ]:
# Confusion Matrix perbandingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, 
    [('Logistic Regression (tuned)', best_lr), ('Random Forest (tuned)', best_rf)]):
    y_pred = model.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=cancer.target_names,
        cmap='Blues', ax=ax
    )
    acc = balanced_accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\nBalanced Acc: {acc:.3f}', fontsize=11, fontweight='bold')

plt.suptitle('Perbandingan Confusion Matrix', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Kurva ROC dan Precision-Recall

### Teori: Mengapa Kita Butuh Kurva Ini?

Accuracy dan F1 dihitung pada threshold default (0.5). Tapi dalam dunia nyata, threshold optimal bergantung pada **biaya kesalahan**:

| Skenario | False Negative lebih mahal | False Positive lebih mahal |
|:---------|:--------------------------|:---------------------------|
| Contoh | Kanker tidak terdeteksi | Email penting masuk spam |
| Strategi | Turunkan threshold (recall ^) | Naikkan threshold (precision ^) |

**ROC Curve**: plot True Positive Rate vs False Positive Rate
- AUC-ROC = 1.0: sempurna
- AUC-ROC = 0.5: sama dengan tebak acak

**Precision-Recall Curve**: lebih informatif saat kelas sangat tidak seimbang
- AP (Average Precision) = area di bawah kurva PR

In [ ]:
# Kurva ROC dan Precision-Recall untuk kedua model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probabilitas prediksi
lr_proba = best_lr.predict_proba(X_test)[:, 1]
rf_proba = best_rf.predict_proba(X_test)[:, 1]

# ROC Curve
RocCurveDisplay.from_predictions(y_test, lr_proba, ax=axes[0], 
                                  name='Logistic Regression')
RocCurveDisplay.from_predictions(y_test, rf_proba, ax=axes[0], 
                                  name='Random Forest')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC=0.5)')
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].legend(loc='lower right')

# Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(y_test, lr_proba, ax=axes[1], 
                                        name='Logistic Regression')
PrecisionRecallDisplay.from_predictions(y_test, rf_proba, ax=axes[1], 
                                        name='Random Forest')
axes[1].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()

# AUC dan AP
print(f'Logistic Regression  -  PR-AUC: {average_precision_score(y_test, lr_proba):.3f}')
print(f'Random Forest        -  PR-AUC: {average_precision_score(y_test, rf_proba):.3f}')
print()
print(' Kurva yang lebih "membengkak" ke kanan-atas = model lebih baik')
print('   PR curve lebih jujur saat kelas tidak seimbang daripada ROC curve')

---

## 7. Feature Importance: Fitur Mana yang Paling Berpengaruh?

### Teori: Permutation Importance

Cara kerja:
1. Hitung skor model pada data test: misalnya 0.95
2. **Acak** satu kolom fitur (misal: `mean radius`)
3. Hitung skor lagi: misalnya 0.70
4. Penurunan skor = 0.95 - 0.70 = 0.25 -> **importance = 0.25**
5. Ulangi untuk setiap fitur

Fitur dengan importance tinggi = model sangat bergantung pada fitur tersebut.

> ** Penting:** Feature importance menunjukkan **asosiasi/pengaruh pada model**, BUKAN sebab-akibat (kausalitas)!

In [ ]:
# Permutation Importance untuk model terbaik (Random Forest)
perm = permutation_importance(
    best_rf, X_test, y_test, 
    scoring='balanced_accuracy',
    n_repeats=15, 
    random_state=RANDOM_STATE, 
    n_jobs=-1
)

# Buat DataFrame dan tampilkan top 15
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

print('Top 15 fitur berdasarkan permutation importance:')
display(importance_df.head(15).round(4))

In [ ]:
# Visualisasi feature importance
top15 = importance_df.head(15).sort_values('importance_mean')

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    top15['feature'], 
    top15['importance_mean'],
    xerr=top15['importance_std'],
    color='steelblue', 
    edgecolor='white',
    capsize=3
)
ax.set_xlabel('Penurunan Balanced Accuracy saat Fitur Diacak')
ax.set_title('Top 15 Fitur: Permutation Importance (Random Forest)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(' Fitur dengan importance tinggi = model sangat bergantung padanya.')
print('   Error bar menunjukkan variasi antar repetisi pengacakan.')
print('   Fitur dengan importance ≤ 0 mungkin tidak berkontribusi / noise.')

---

## 8. Perbandingan Sistematis Semua Model

Mari kita bandingkan SEMUA model yang sudah kita pelajari dari Level Medium sampai sekarang.

In [ ]:
# Perbandingan lengkap semua model menggunakan cross-validation
all_pipelines = {
    'Baseline (majority)': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
    'KNN (K=5)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Decision Tree': Pipeline([
        ('model', DecisionTreeClassifier(max_depth=5, class_weight='balanced', 
                                         random_state=RANDOM_STATE))
    ]),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=3000, class_weight='balanced', 
                                     random_state=RANDOM_STATE))
    ]),
    'LR (tuned)': best_lr,
    'Random Forest (tuned)': best_rf,
}

comparison_results = []
for name, model in all_pipelines.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='balanced_accuracy')
    comparison_results.append({
        'Model': name,
        'CV Mean': scores.mean(),
        'CV Std': scores.std(),
        'CV Scores': scores
    })

comp_df = pd.DataFrame(comparison_results).set_index('Model')
print('Perbandingan Cross-Validation (Balanced Accuracy):')
display(comp_df[['CV Mean', 'CV Std']].sort_values('CV Mean', ascending=False).round(4))

In [ ]:
# Box plot perbandingan
fig, ax = plt.subplots(figsize=(12, 5))

box_data = [r['CV Scores'] for r in comparison_results]
box_labels = [r['Model'] for r in comparison_results]

bp = ax.boxplot(box_data, patch_artist=True, vert=True)
ax.set_xticklabels(box_labels)  # kompatibel dengan matplotlib lama & baru
colors = ['#d3d3d3', '#66b3ff', '#99ff99', '#ffcc99', '#ff9999', '#c299ff']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_ylabel('Balanced Accuracy')
ax.set_title('Perbandingan Model: CV Balanced Accuracy', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=25)
ax.set_ylim(0.4, 1.05)
plt.tight_layout()
plt.show()

---

## Latihan dan Quiz

### Quiz Teori

1. Apa perbedaan antara **accuracy** dan **balanced accuracy**? Kapan masing-masing lebih tepat?
2. Mengapa kita menggunakan `class_weight='balanced'` pada Logistic Regression?
3. Jelaskan apa yang terjadi jika kita menurunkan threshold prediksi dari 0.5 ke 0.2. Apa dampaknya pada precision dan recall?
4. Apa perbedaan antara ROC curve dan Precision-Recall curve? Kapan masing-masing lebih informatif?
5. Mengapa feature importance bukan berarti **kausalitas**? Berikan contoh.
6. Apa risiko jika kita melakukan tuning pada data test?

### Jawaban Quiz

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

In [ ]:
# LATIHAN 1: GridSearchCV untuk KNN
# ──────────────────────────────────
# Tugas:
#  a) Buat pipeline KNN dengan scaler
#  b) Definisikan param_grid: n_neighbors=[1,3,5,7,9,11,15,21]
#  c) Jalankan GridSearchCV dengan scoring='balanced_accuracy'
#  d) Tampilkan parameter terbaik dan skor
#  e) Evaluasi pada data test
#
# Tulis kode Anda di bawah:



In [ ]:
# LATIHAN 2: Pengaruh class_weight
# ─────────────────────────────────
# Tugas:
#  a) Latih Logistic Regression TANPA class_weight='balanced'
#  b) Latih Logistic Regression DENGAN class_weight='balanced'
#  c) Bandingkan classification report keduanya
#  d) Perhatikan perbedaan recall untuk kelas malignant
#  e) Jelaskan mengapa class_weight penting untuk masalah medis
#
# Tulis kode Anda di bawah:



In [ ]:
# LATIHAN 3: Membandingkan scoring metrics pada GridSearchCV
# ──────────────────────────────────────────────────────────
# Tugas:
#  a) Jalankan GridSearchCV untuk Random Forest dengan scoring='accuracy'
#  b) Jalankan GridSearchCV lagi dengan scoring='balanced_accuracy'
#  c) Apakah parameter terbaik berbeda? Mengapa?
#
# Tulis kode Anda di bawah:



---

## Checklist Sebelum Lanjut ke Level Extra High

Pastikan Anda sudah bisa:

- [ ] Menjelaskan apa itu hyperparameter dan mengapa perlu di-tune
- [ ] Menggunakan GridSearchCV untuk mencari parameter terbaik
- [ ] Memahami dan menangani kelas tidak seimbang (`class_weight`, balanced accuracy)
- [ ] Membaca kurva ROC dan Precision-Recall
- [ ] Menggunakan Random Forest dan memahami konsep ensemble
- [ ] Melakukan permutation importance dan menginterpretasinya
- [ ] Membandingkan beberapa model secara sistematis
- [ ] Mengevaluasi pada data test HANYA SATU KALI di akhir

### Lanjut ke `04_level_extra_high.ipynb` ->